# G1：ROS 2 话题与服务 —— 分布式通信
# G1: ROS 2 Topics & Services — Distributed Communication

## 实验目标 | Objectives
- 理解 ROS 2 的话题（topics）与服务（services）机制
  Understand ROS 2 topics and services
- 掌握 ROS 2 在 IEV CloudLab 架构中的定位
  Know where ROS 2 fits in the IEV CloudLab architecture

## 背景知识 | Background
**ROS 2**：机器人/自动驾驶的事实标准中间件。
- **话题（Topic）**：异步发布/订阅，如 `/vehicle/speed`（车辆速度广播）
- **服务（Service）**：同步请求/响应，如 `/get_map`（请求地图）
- **动作（Action）**：长时间任务，如导航

> **架构定位**（重要）：IEV CloudLab 中 ROS 2 **不做车辆闭环控制**
> （A1/B3 的控制器直调仿真器 API，延迟更低）。
> ROS 2 用于：多车状态广播、参数服务、数据记录可视化（如 foxglove_bridge）。


In [ ]:
import os
print('ROS 2 Jazzy 可用:', os.path.exists('/opt/ros/jazzy'))
# ROS 2 环境（notebook 中演示概念，完整节点需终端运行）
print('\\nROS 2 核心概念:')
print('  - Topic: 异步发布/订阅（如 /vehicle/speed）')
print('  - Service: 同步请求/响应（如 /get_map）')
print('  - Action: 长时间任务（如导航）')


## 1. 话题通信模式（概念演示）

In [ ]:
# 用 Python 模拟 ROS 2 话题的发布/订阅模式
from collections import defaultdict
import time

class MiniROS:
    """极简 ROS 2 话题模拟"""
    def __init__(self):
        self.subs = defaultdict(list)
    def publish(self, topic, msg):
        for cb in self.subs[topic]:
            cb(msg)
    def subscribe(self, topic, callback):
        self.subs[topic].append(callback)

ros = MiniROS()
received = []
ros.subscribe('/vehicle/speed', lambda m: received.append(m))
# 模拟车辆速度发布
for v in [15.0, 14.5, 14.0]:
    ros.publish('/vehicle/speed', {'v': v, 't': time.time()})
print(f'收到 {len(received)} 条速度消息: {[m["v"] for m in received]}')
assert len(received) == 3
print('✅ 话题通信验证通过！')


## 2. 在 IEV CloudLab 中的定位
- **不用 ROS 做闭环**：A1/B3 的控制器直接调仿真器 API，延迟更低
- **用 ROS 做什么**：多车协同的状态广播、参数服务、数据记录
- **foxglove_bridge**：WebSocket 桥接，可视化（见设计文档 §4）

## 🏆 挑战任务
1. 在终端启动 `ros2 run demo_nodes_cpp talker`，用 `ros2 topic echo` 观察。
2. 设计一个 `/platoon/status` 话题的消息格式。